In [23]:
import pandas as pd
data = pd.read_csv("sauto_dataset.csv", sep=',')


data.loc[data["pohon"].isna(), "pohon"] = "nezadano"
# maska_ke_smazani = (data['palivo'] == 'elektro') & ((data['kapacita baterie'] == 0) | (data['kapacita baterie'].isna()))
# data = data[~maska_ke_smazani]

indexy_elektro = data[(data['palivo'] == 'Elektro') & (data['kapacita_baterie'] == 0)].index
indexy_none = data[(data['znacka'].isna()) | (data['model'].isna()) | (data['cena'].isna()) | (data['rok'].isna()) | (data['stav'].isna()) | (data['najeto'].isna()) | (data['vykon'].isna()) | (data['palivo'].isna()) | (data['prevodovka'].isna())].index

vsechny_ke_smazani = indexy_elektro.union(indexy_none)
# vsechny_ke_smazani = indexy_elektro
data = data.drop(vsechny_ke_smazani)


age_of_car = 2026 - data['rok']
mileage_per_year = data['najeto'] / age_of_car
data['age_of_car'] = age_of_car
data['mileage_per_year'] = mileage_per_year




# string_features = ['znacka', 'model', 'stav', 'palivo', 'prevodovka', 'pohon']
# mapping = {}
# for feature in string_features:
#     data[feature] = data[feature].astype('category')
#     mapping[feature] = list(data[feature].cat.categories)
#     data[feature] = data[feature].cat.codes

# print(mapping['znacka'])
# print(mapping['stav'])
# print(mapping['palivo'])
# print(mapping['prevodovka'])
# print(mapping['pohon'])
#
# kod = mapping["znacka"].index('Tesla')
# print(f"znacka: ",kod)
# kod = mapping["model"].index('Model 3')
# print(f"model: ",kod)



# maska = (data["palivo"] == "Elektro") & (data["kapacita_baterie"] == 0)
# print(data[maska])# data

# idk = (data["palivo"] == "Elektro")
# print(data[idk])
# print(data["znacka"].unique())

data

,znacka,model,cena,rok,stav,najeto,vykon,palivo,kapacita_baterie,prevodovka,pohon,age_of_car,mileage_per_year
0,Škoda,Superb,340000.0,2016.0,Ojeté,228069,140.0,Nafta,0,Automatická,nezadano,10.0,22806.900000
1,Dacia,Duster,180000.0,2017.0,Ojeté,149609,80.0,Nafta,0,Manuální,4x4,9.0,16623.222222
2,Škoda,Octavia,100000.0,2010.0,Ojeté,211625,77.0,Benzín,0,Manuální,nezadano,16.0,13226.562500
4,Mercedes-Benz,Třídy C,899000.0,2021.0,Ojeté,95000,162.0,Nafta,0,Automatická,nezadano,5.0,19000.000000
5,Volkswagen,Tiguan,460000.0,2018.0,Ojeté,145317,110.0,Nafta,0,Automatická,nezadano,8.0,18164.625000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
33894,Volkswagen,T-Roc,599000.0,2021.0,Ojeté,46585,110.0,Benzín,0,Automatická,Pohon předních kol,5.0,9317.000000
33895,Škoda,Octavia,468900.0,2021.0,Ojeté,92769,110.0,Benzín,0,Manuální,Pohon předních kol,5.0,18553.800000
33896,Škoda,Scala,377900.0,2021.0,Ojeté,72402,81.0,Benzín,0,Manuální,Pohon předních kol,5.0,14480.400000
33897,Škoda,Octavia,457900.0,2021.0,Ojeté,127575,110.0,Benzín,0,Automatická,Pohon předních kol,5.0,25515.000000


In [25]:
import numpy as np
from sklearn.model_selection import train_test_split
from catboost import CatBoostRegressor


input_features = ["znacka", "model", "rok", "stav", "najeto", "vykon", "palivo", "kapacita_baterie", "prevodovka", "pohon"
    , "age_of_car", "mileage_per_year"
                  ]
target_feature = 'cena'
cat_features = ["znacka", "model", "stav", "palivo", "prevodovka", "pohon"]

data[cat_features] = data[cat_features].astype(str)
X_train, X_test, y_train, y_test = train_test_split(data[input_features], data[target_feature], test_size=0.1, random_state=42)


model = CatBoostRegressor(
    iterations=500,
    learning_rate=0.1,
    depth=10,
    cat_features=cat_features,
    random_state=42,
    verbose=100,
    task_type="GPU",
)

# log_train = np.log(y_train)
model.fit(X_train, y_train )
          # , eval_set=(X_test, y_test))

# y_pred = np.exp(model.predict(X_test))
y_pred = model.predict(X_test)


y_pred = np.clip(y_pred, a_min=5000, a_max=None)

0:	learn: 1265871.6334126	total: 163ms	remaining: 1m 21s
100:	learn: 528453.9962750	total: 12.9s	remaining: 50.9s
200:	learn: 477587.5556235	total: 25.1s	remaining: 37.3s
300:	learn: 453410.2735113	total: 36.9s	remaining: 24.4s
400:	learn: 402184.0676130	total: 49.5s	remaining: 12.2s
499:	learn: 339439.4053117	total: 1m 1s	remaining: 0us


In [21]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mse = mean_squared_error(y_test, y_pred)

print("-" * 30)
print(f"MAE: {mae:,.0f} Kč".replace(',', ' '))
print(f"RMSE: {rmse:,.0f} Kč".replace(',', ' '))
print(f"MSE: {mse:,.0f} Kč".replace(',', ' '))
print("-" * 30)

------------------------------
MAE: 98 267 Kč
RMSE: 326 228 Kč
MSE: 106 424 503 491 Kč
------------------------------


In [21]:
print(model.tree_.threshold)

[ 4.91499996e+01  2.01650000e+03  2.01250000e+03  2.00950000e+03
  3.77999992e+01  2.00750000e+03 -2.00000000e+00 -2.00000000e+00
  2.00650000e+03 -2.00000000e+00 -2.00000000e+00  4.75000000e+01
  2.01150000e+03 -2.00000000e+00 -2.00000000e+00 -2.00000000e+00
  2.01550000e+03  3.18510000e+04  3.08870000e+04 -2.00000000e+00
 -2.00000000e+00  1.05000000e+01 -2.00000000e+00 -2.00000000e+00
  4.67500000e+01  4.19499989e+01 -2.00000000e+00 -2.00000000e+00
  8.00000000e+00 -2.00000000e+00 -2.00000000e+00  1.80000001e+00
  4.51999989e+01  5.00000000e-01  4.12000008e+01 -2.00000000e+00
 -2.00000000e+00  4.01000004e+01 -2.00000000e+00 -2.00000000e+00
  1.55000001e+00  1.50000000e+00 -2.00000000e+00 -2.00000000e+00
  1.36050000e+03 -2.00000000e+00 -2.00000000e+00  2.01850000e+03
  4.25000000e+01  3.97820000e+04 -2.00000000e+00 -2.00000000e+00
  2.41120000e+04 -2.00000000e+00 -2.00000000e+00  4.50000000e+00
  4.18500004e+01 -2.00000000e+00 -2.00000000e+00  7.00000000e+00
 -2.00000000e+00 -2.00000

In [38]:
import pickle

with open("model_sautoV5_catBoost.dat", "wb") as soubor:
     pickle.dump(model, soubor)

In [26]:
model.save_model("model_sautoV6_catBoost(500).cbm")